# 01 数据理解与口径核查

目标：确认数据粒度、规模、缺失情况和需要业务确认的字段。数据是一人一行的用户汇总表，不能直接回答活动是否带来增量购买。

In [1]:
from pathlib import Path
import sys
# 兼容从项目根目录或 code 目录打开。
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'code/analysis.py').exists())
if str(ROOT / 'code') not in sys.path:
    sys.path.insert(0, str(ROOT / 'code'))
import pandas as pd
from IPython.display import display, Image


In [2]:
from analysis import load_data, audit_data
raw = load_data()
raw.head()

,User_ID,Age,Gender,Location,Income,Interests,Last_Login_Days_Ago,Purchase_Frequency,Average_Order_Value,Total_Spending,Product_Category_Preference,Time_Spent_on_Site_Minutes,Pages_Viewed,Newsletter_Subscription
0,#1,56,Male,Suburban,38037,Sports,5,7,18,2546,Books,584,38,True
1,#2,46,Female,Rural,103986,Technology,15,7,118,320,Electronics,432,40,False
2,#3,32,Female,Suburban,101942,Sports,28,1,146,3766,Apparel,306,1,True
3,#4,60,Female,Suburban,71612,Fashion,18,3,163,4377,Apparel,527,29,False
4,#5,25,Male,Suburban,49725,Travel,2,5,141,4502,Health & Beauty,53,10,True


## 样本规模与字段
CSV 中的导出行号不算业务字段。检查唯一用户数，而不是只看行数。

In [3]:
print('行数', len(raw), '业务字段数', raw.shape[1])
print('唯一用户数', raw.User_ID.nunique())
raw.dtypes.to_frame('类型')

行数 1000 业务字段数 14
唯一用户数 1000


,类型
User_ID,str
Age,int64
Gender,str
Location,str
Income,int64
Interests,str
Last_Login_Days_Ago,int64
Purchase_Frequency,int64
Average_Order_Value,int64
Total_Spending,int64


## 口径核查
购买频次为 0、累计消费大于 0 不必然是错误；如果统计周期不同，两者可以同时成立。先标记，再研究结论对这些记录的敏感性。

In [4]:
audit = audit_data(raw)
audit

,检查项,结果
0,用户数,1000
1,业务字段数,14
2,重复用户ID数,0
3,缺失单元格数,0
4,待核对用户数,85
5,最近登录天数最小值,1
6,最近登录天数最大值,29
7,邮件订阅人数,507


In [5]:
marked = raw.loc[(raw.Purchase_Frequency == 0) & (raw.Total_Spending > 0)]
print('待核对用户', len(marked))
marked[['User_ID','Purchase_Frequency','Total_Spending']].head(10)

待核对用户 85


,User_ID,Purchase_Frequency,Total_Spending
27,#28,0,533
40,#41,0,1273
49,#50,0,3915
53,#54,0,3410
66,#67,0,3164
81,#82,0,2184
94,#95,0,4250
111,#112,0,1268
116,#117,0,2967
117,#118,0,3616


## 字段取值与兴趣映射
如果兴趣和商品类别不使用同一套名称，不能直接比较字符串是否相同并将其解释成匹配程度。

In [6]:
print('最近登录天数范围', raw.Last_Login_Days_Ago.min(), raw.Last_Login_Days_Ago.max())
print('兴趣标签', sorted(raw.Interests.unique()))
print('偏好品类', sorted(raw.Product_Category_Preference.unique()))
print('直接重名的类别', set(raw.Interests) & set(raw.Product_Category_Preference))

最近登录天数范围 1 29
兴趣标签 ['Fashion', 'Food', 'Sports', 'Technology', 'Travel']
偏好品类 ['Apparel', 'Books', 'Electronics', 'Health & Beauty', 'Home & Kitchen']
直接重名的类别 set()


## 本阶段结论
- 1,000 位唯一用户，14 个业务字段。
- 85 位用户需要确认消费统计口径。
- 最近登录为 1–29 天；它不是最近购买。
- 没有发券、随机分组或活动后订单记录。

检查结果已保存到 `outputs/audit/`。